# DCCN Lab Programs — Short, Runnable Python Codes

**How to use:** Run cells one at a time. The waveform and modulation programs use `matplotlib`; if needed, install it with `pip install matplotlib`.

**Socket programs:** These examples run a client and server on `localhost` using threads, so you can demonstrate them in one notebook without opening two terminals. For a real client/server setup, use the same server/client logic on separate machines and change the host address.

## 1. Dijkstra's Algorithm (Lab-2)

In [1]:
import heapq

def dijkstra(graph, start):
    dist = {v: float("inf") for v in graph}
    dist[start] = 0
    pq = [(0, start)]
    while pq:
        d, u = heapq.heappop(pq)
        if d != dist[u]:
            continue
        for v, w in graph[u].items():
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist

graph = {
    "A": {"B": 4, "C": 1}, "B": {"A": 4, "C": 2, "D": 1},
    "C": {"A": 1, "B": 2, "D": 5}, "D": {"B": 1, "C": 5}
}
start = input("Starting node (A/B/C/D): ").strip().upper() or "A"
print("Shortest distances:", dijkstra(graph, start))

Shortest distances: {'A': 0, 'B': 3, 'C': 1, 'D': 4}


## 2. Digital waveforms: NRZ-L, NRZ-I, AMI and Pseudoternary

In [ ]:
import matplotlib.pyplot as plt

bits = input("Enter bit string (e.g. 101100): ").strip()
if not bits or any(b not in "01" for b in bits):
    raise ValueError("Enter only 0 and 1")

def draw(levels, title):
    x, y = [], []
    for i, level in enumerate(levels):
        x += [i, i + 1]; y += [level, level]
    plt.step(x, y, where="post")
    plt.xticks(range(len(bits) + 1)); plt.yticks([-1, 0, 1])
    plt.grid(True); plt.xlim(0, len(bits)); plt.title(title)

nrz_l = [1 if b == "0" else -1 for b in bits]
nrz_i, level = [], -1
for b in bits:
    if b == "1": level *= -1
    nrz_i.append(level)
ami, level = [], -1
for b in bits:
    if b == "1": level *= -1; ami.append(level)
    else: ami.append(0)
pseudo, level = [], -1
for b in bits:
    if b == "0": level *= -1; pseudo.append(level)
    else: pseudo.append(0)

fig, ax = plt.subplots(4, 1, figsize=(10, 7))
for a, vals, title in zip(ax, [nrz_l, nrz_i, ami, pseudo],
                          ["NRZ-L (0=+1, 1=-1)", "NRZ-I", "Bipolar AMI", "Pseudoternary"]):
    plt.sca(a); draw(vals, title)
plt.tight_layout(); plt.show()

## 3. Manchester and Differential Manchester

In [ ]:
import matplotlib.pyplot as plt

bits = input("Enter bit string (e.g. 101100): ").strip()
if not bits or any(b not in "01" for b in bits): raise ValueError("Use only 0 and 1")
man, diff = [], []
prev = 1
for b in bits:
    man += ([1, -1] if b == "0" else [-1, 1])
    if b == "0": prev *= -1
    diff += [prev, -prev]
    prev = -prev

fig, ax = plt.subplots(2, 1, figsize=(10, 4))
for a, vals, title in zip(ax, [man, diff], ["Manchester (0=low-to-high)", "Differential Manchester (0=transition at start)"]):
    a.step([i/2 for i in range(len(vals)+1)], vals + [vals[-1]], where="post")
    a.set_xticks(range(len(bits)+1)); a.set_yticks([-1, 1]); a.grid(True); a.set_title(title)
    a.set_xlim(0, len(bits))
plt.tight_layout(); plt.show()

## 4. IP Addressing: Classes A, B, C, D and E

In [ ]:
import ipaddress

ip = ipaddress.IPv4Address(input("Enter IPv4 address: ").strip())
first = int(str(ip).split(".")[0])
if 1 <= first <= 126: cls, default = "A", "255.0.0.0"
elif first == 127: cls, default = "A (loopback range)", "255.0.0.0"
elif 128 <= first <= 191: cls, default = "B", "255.255.0.0"
elif 192 <= first <= 223: cls, default = "C", "255.255.255.0"
elif 224 <= first <= 239: cls, default = "D (multicast)", "Not applicable"
else: cls, default = "E (experimental/reserved)", "Not applicable"
print("IP:", ip, "\nClass:", cls, "\nDefault mask:", default)

## 5. Convert IPv4 Address to 32-Bit Binary

In [ ]:
import ipaddress

ip = ipaddress.IPv4Address(input("Enter IPv4 address: ").strip())
binary = ".".join(f"{int(octet):08b}" for octet in str(ip).split("."))
print("32-bit address:", binary)
print("Without dots:", binary.replace(".", ""))

## 6. Digital-to-Analog Conversion: ASK, PSK and FSK

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

bits = input("Enter bit string (e.g. 10110): ").strip()
if not bits or any(b not in "01" for b in bits): raise ValueError("Use only 0 and 1")
t = np.linspace(0, len(bits), 2000)
ask, psk, fsk = [], [], []
for x in t:
    i = min(int(x), len(bits)-1); local = x - i
    b = int(bits[i])
    ask.append((1 if b else 0.25) * np.sin(2*np.pi*5*local))
    psk.append(np.sin(2*np.pi*5*local + (np.pi if b else 0)))
    fsk.append(np.sin(2*np.pi*(8 if b else 3)*local))
fig, ax = plt.subplots(3, 1, figsize=(10, 6))
for a, y, title in zip(ax, [ask, psk, fsk], ["ASK", "BPSK", "BFSK"]):
    a.plot(t, y); a.set_title(title); a.grid(True); a.set_xlim(0, len(bits))
plt.tight_layout(); plt.show()

## 7. Digital-to-Analog Conversion: QAM and QPSK (illustrative)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

bits = input("Enter an even-length bit string (e.g. 10110010): ").strip()
if not bits or len(bits) % 2 or any(b not in "01" for b in bits):
    raise ValueError("Enter an even-length string containing only 0 and 1")
pairs = [bits[i:i+2] for i in range(0, len(bits), 2)]
# Gray-coded QPSK phases
phase_map = {"00": np.pi/4, "01": 3*np.pi/4, "11": 5*np.pi/4, "10": 7*np.pi/4}
samples = [phase_map[p] for p in pairs]
plt.figure(figsize=(5,5))
plt.scatter(np.cos(samples), np.sin(samples))
for p, ph in zip(pairs, samples): plt.annotate(p, (np.cos(ph), np.sin(ph)))
plt.axhline(0); plt.axvline(0); plt.grid(True); plt.axis("equal"); plt.title("QPSK constellation")
plt.show()
# Simple 16-QAM constellation
levels = [-3, -1, 1, 3]
points = [(i, q) for i in levels for q in levels]
plt.figure(figsize=(5,5))
plt.scatter([p[0] for p in points], [p[1] for p in points])
plt.grid(True); plt.axhline(0); plt.axvline(0); plt.axis("equal"); plt.title("16-QAM constellation")
plt.show()

## 8. HDB3 and B8ZS (simplified substitution demonstration)

In [ ]:
def ami(bits):
    out, last = [], -1
    for b in bits:
        if b == "1": last *= -1; out.append("+" if last == 1 else "-")
        else: out.append("0")
    return out

bits = input("Enter bit string (e.g. 100000000): ").strip()
if not bits or any(b not in "01" for b in bits): raise ValueError("Use only 0 and 1")
# Replace zero runs: B8ZS: 000VB0VB; HDB3: 000V or B00V.
def b8zs(s):
    out, last = [], -1
    i = 0
    while i < len(s):
        if s[i:i+8] == "00000000":
            # V has same polarity as prior pulse; B alternates from prior pulse.
            B = -last; V = last
            out += ["0","0","0", "B", "V", "0", "V", "B"]
            # Pattern labels demonstrate substitution; update last to final B.
            last = B; i += 8
        else:
            if s[i] == "1": last *= -1; out.append("+" if last == 1 else "-")
            else: out.append("0")
            i += 1
    return out
def hdb3(s):
    out, last, count = [], -1, 0
    i = 0
    while i < len(s):
        if s[i:i+4] == "0000":
            # Standard substitution depends on pulse parity since previous violation.
            if count % 2 == 0:
                # B00V: balancing pulse then violation pulse
                B = -last; V = B
                out += ["B", "0", "0", "V"]; last = V
            else:
                V = last
                out += ["0", "0", "0", "V"]; last = V
            count = 0; i += 4
        else:
            if s[i] == "1": last *= -1; out.append("+" if last == 1 else "-"); count += 1
            else: out.append("0")
            i += 1
    return out
print("B8ZS:", " ".join(b8zs(bits)))
print("HDB3:", " ".join(hdb3(bits)))
print("Note: B/V are substitution pulse labels; this is a compact lab demonstration.")

## 9. Hamming Code (even parity, single-bit correction)

In [ ]:
data = input("Enter data bits (e.g. 1011): ").strip()
if not data or any(b not in "01" for b in data): raise ValueError("Use only 0 and 1")
m = len(data); r = 0
while 2**r < m + r + 1: r += 1
n = m + r
arr = [0] * (n + 1)
j = 0
for i in range(1, n+1):
    if i & (i-1): arr[i] = int(data[j]); j += 1
for p in range(r):
    pos = 2**p
    arr[pos] = sum(arr[i] for i in range(1, n+1) if i & pos and i != pos) % 2
print("Hamming code:", "".join(map(str, arr[1:])))
received = input("Enter received code (press Enter to use generated code): ").strip()
if not received: received = "".join(map(str, arr[1:]))
if len(received) != n or any(b not in "01" for b in received): raise ValueError("Received code length must match")
a = [0] + list(map(int, received)); syndrome = 0
for p in range(r):
    pos = 2**p
    if sum(a[i] for i in range(1, n+1) if i & pos) % 2: syndrome += pos
if syndrome:
    print("Error at position:", syndrome)
    a[syndrome] ^= 1
    print("Corrected code:", "".join(map(str, a[1:])))
else: print("No error detected.")

## 10. Stop-and-Wait ARQ

In [ ]:
import random
frames = int(input("Number of frames: "))
for i in range(1, frames+1):
    print(f"Sending frame {i}")
    if random.random() < 0.25:
        print("ACK lost; retransmitting frame", i)
        print("ACK received for frame", i)
    else:
        print("ACK received for frame", i)
print("Transmission complete.")

## 11. Sliding Window ARQ (simple Go-Back-N simulation)

In [ ]:
frames = int(input("Number of frames: "))
window = int(input("Window size: "))
if frames < 1 or window < 1: raise ValueError("Values must be positive")
i = 1
while i <= frames:
    end = min(i + window - 1, frames)
    print(f"Sending window: frames {i} to {end}")
    lost = input(f"Was frame {end} lost? (y/n): ").strip().lower()
    if lost == "y":
        print(f"Timeout: retransmit from frame {end}")
        print(f"Frame {end} acknowledged")
    else:
        print(f"ACK received for frames {i} to {end}")
    i = end + 1
print("Transmission complete.")

## 12. CRC with single-bit error detection

In [ ]:
def divide(bits, poly):
    bits = list(map(int, bits))
    p = list(map(int, poly))
    for i in range(len(bits)-len(p)+1):
        if bits[i]:
            for j in range(len(p)): bits[i+j] ^= p[j]
    return "".join(map(str, bits[-(len(p)-1):])) if len(p) > 1 else ""
msg = input("Message bits: ").strip()
poly = input("Generator polynomial bits (e.g. 1101): ").strip()
if any(c not in "01" for c in msg+poly) or len(poly) < 2 or poly[0] != "1": raise ValueError("Invalid bits/polynomial")
remainder = divide(msg + "0"*(len(poly)-1), poly)
frame = msg + remainder
print("CRC:", remainder, "\nTransmitted codeword:", frame)
received = list(frame)
pos = int(input(f"Bit position to flip (1-{len(frame)}, 0 for none): "))
if pos:
    if not 1 <= pos <= len(frame): raise ValueError("Position out of range")
    received[pos-1] = "1" if received[pos-1] == "0" else "0"
check = divide("".join(received), poly)
print("Received:", "".join(received))
print("Error detected" if "1" in check else "No error detected", "(remainder:", check + ")")

## 13. CRC-12, CRC-16 and CRC-32 for a string dataset

In [ ]:
import zlib
import binascii

dataset = input("Enter message/string: ").encode()
# Standard commonly used CRC variants
crc16_poly = 0x1021
crc = 0xFFFF
for byte in dataset:
    crc ^= byte << 8
    for _ in range(8):
        crc = ((crc << 1) ^ crc16_poly) & 0xFFFF if crc & 0x8000 else (crc << 1) & 0xFFFF
crc12 = 0
for byte in dataset:
    crc12 ^= byte << 4
    for _ in range(8):
        crc12 = ((crc12 << 1) ^ 0x80F) & 0xFFF if crc12 & 0x800 else (crc12 << 1) & 0xFFF
print(f"CRC-12 (CRC-12/3GPP-style polynomial approximation): {crc12:03X}")
print(f"CRC-16/CCITT-FALSE: {crc:04X}")
print(f"CRC-32/IEEE: {zlib.crc32(dataset) & 0xFFFFFFFF:08X}")

## 14. Subnetting and Supernetting

In [ ]:
import ipaddress

network = ipaddress.ip_network(input("Enter network (e.g. 192.168.1.0/24): ").strip(), strict=False)
new_prefix = int(input("New prefix length (e.g. 26 to split /24): "))
if not network.prefixlen <= new_prefix <= network.max_prefixlen: raise ValueError("New prefix must be >= current prefix")
subs = list(network.subnets(new_prefix=new_prefix))
print("Subnets:")
for s in subs: print(s, "hosts:", max(s.num_addresses-2, 0))
print("\nSupernet (one level broader):", network.supernet())

## 15. Socket Programming: Basic Client–Server Model

In [ ]:
import socket, threading

def server():
    with socket.socket() as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(("127.0.0.1", 0)); s.listen()
        port_box.append(s.getsockname()[1]); ready.set()
        conn, _ = s.accept()
        with conn: conn.sendall(b"Hello from server")

port_box, ready = [], threading.Event()
thread = threading.Thread(target=server, daemon=True); thread.start(); ready.wait()
with socket.socket() as c:
    c.connect(("127.0.0.1", port_box[0]))
    print("Client received:", c.recv(1024).decode())
thread.join(timeout=1)

## 16. TCP: Send and Receive a Message Both Ways

In [ ]:
import socket, threading

def server():
    with socket.socket() as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(("127.0.0.1", 0)); s.listen(); port[0] = s.getsockname()[1]; ready.set()
        conn, _ = s.accept()
        with conn:
            msg = conn.recv(1024).decode()
            print("Server received:", msg)
            conn.sendall(b"Reply from server")

port, ready = [0], threading.Event()
t = threading.Thread(target=server, daemon=True); t.start(); ready.wait()
with socket.socket() as c:
    c.connect(("127.0.0.1", port[0]))
    c.sendall(b"Hello from client")
    print("Client received:", c.recv(1024).decode())
t.join(timeout=1)

## 17. TCP: Send/Receive a Text File (character stream)

In [ ]:
import socket, threading

content = "DCCN text file demo\nSent using TCP character encoding."
def server():
    with socket.socket() as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(("127.0.0.1", 0)); s.listen(); port[0] = s.getsockname()[1]; ready.set()
        conn, _ = s.accept()
        with conn:
            received = conn.recv(4096).decode("utf-8")
            print("Server received text:", received)
            conn.sendall("Text file reply from server".encode("utf-8"))

port, ready = [0], threading.Event()
t = threading.Thread(target=server, daemon=True); t.start(); ready.wait()
with socket.socket() as c:
    c.connect(("127.0.0.1", port[0]))
    c.sendall(content.encode("utf-8"))
    print("Client received:", c.recv(4096).decode("utf-8"))
t.join(timeout=1)

## 18. TCP: Send/Receive Any File (binary stream)

In [ ]:
import socket, threading, tempfile, os

# Create a small binary file for the demonstration.
source = tempfile.NamedTemporaryFile(delete=False)
source.write(bytes([0, 1, 2, 3, 127, 128, 255])); source.close()
with open(source.name, "rb") as f: payload = f.read()

def recv_all(conn):
    chunks = []
    while True:
        data = conn.recv(1024)
        if not data: break
        chunks.append(data)
    return b"".join(chunks)

def server():
    with socket.socket() as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(("127.0.0.1", 0)); s.listen(); port[0] = s.getsockname()[1]; ready.set()
        conn, _ = s.accept()
        with conn:
            received = recv_all(conn)
            with open(received_path, "wb") as f: f.write(received)
            print("Server received bytes:", len(received))
            conn2_data = b"Binary reply from server\x00\xff"
            # Use a length prefix so the client knows when the response ends.
            conn.sendall(len(conn2_data).to_bytes(4, "big") + conn2_data)

port, ready = [0], threading.Event()
received_path = tempfile.mktemp(prefix="dccn_received_")
t = threading.Thread(target=server, daemon=True); t.start(); ready.wait()
with socket.socket() as c:
    c.connect(("127.0.0.1", port[0]))
    c.sendall(payload); c.shutdown(socket.SHUT_WR)
    header = c.recv(4)
    length = int.from_bytes(header, "big")
    reply = b""
    while len(reply) < length: reply += c.recv(length-len(reply))
    print("Client received binary reply:", reply)
print("Received file saved at:", received_path)
os.unlink(source.name)
t.join(timeout=1)